In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StarSchema") \
    .config(
        "spark.hadoop.hive.metastore.uris",
        "thrift://docker-hive-hive-metastore-1:9083"
    ) \
    .enableHiveSupport() \
    .getOrCreate()

In [3]:
silver_df = spark.read.parquet(
    "hdfs://docker-hive-namenode-1:8020/data-lake/silver/transaksi"
)

silver_df.show()

+---+----------+------+----------+--------+------+-------+-------+
| id|   tanggal|produk|  kategori|    kota|jumlah|  harga|revenue|
+---+----------+------+----------+--------+------+-------+-------+
|  1|2026-05-01|Laptop|Elektronik| Jakarta|     1|8000000|8000000|
|  2|2026-05-01| Mouse|Elektronik| Bandung|     2| 150000| 300000|
|  3|2026-05-02|  Buku|   Edukasi|Surabaya|     3|  75000| 225000|
|  4|2026-05-02|Laptop|Elektronik| Jakarta|     1|8000000|8000000|
|  5|2026-05-03|Pulpen|   Edukasi|  Kediri|    10|   5000|  50000|
|  6|2026-05-03|Laptop|Elektronik| Jakarta|     1|8000000|8000000|
|  7|2026-05-04| Mouse|Elektronik| Bandung|     1| 150000| 150000|
|  8|2026-05-04|  Buku|   Edukasi|Surabaya|     2|  75000| 150000|
+---+----------+------+----------+--------+------+-------+-------+



In [4]:
from pyspark.sql.functions import monotonically_increasing_id

dim_produk = (
    silver_df
    .select("produk", "kategori")
    .distinct()
)

dim_produk = dim_produk.withColumn(
    "produk_key",
    monotonically_increasing_id() + 1
)

dim_produk.show()

+------+----------+----------+
|produk|  kategori|produk_key|
+------+----------+----------+
| Mouse|Elektronik|         1|
|Pulpen|   Edukasi|         2|
|  Buku|   Edukasi|         3|
|Laptop|Elektronik|         4|
+------+----------+----------+



In [5]:
dim_kota = (
    silver_df
    .select("kota")
    .distinct()
)

dim_kota = dim_kota.withColumn(
    "kota_key",
    monotonically_increasing_id() + 1
)

dim_kota.show()

+--------+--------+
|    kota|kota_key|
+--------+--------+
|  Kediri|       1|
| Jakarta|       2|
|Surabaya|       3|
| Bandung|       4|
+--------+--------+



In [6]:
from pyspark.sql.functions import (
    year,
    month,
    quarter,
    dayofmonth,
    date_format
)

dim_tanggal = (
    silver_df
    .select("tanggal")
    .distinct()
)

dim_tanggal = (
    dim_tanggal
    .withColumn("tahun", year("tanggal"))
    .withColumn("bulan", month("tanggal"))
    .withColumn("kuartal", quarter("tanggal"))
    .withColumn("hari", dayofmonth("tanggal"))
    .withColumn(
        "nama_bulan",
        date_format("tanggal", "MMMM")
    )
)

dim_tanggal.show()

+----------+-----+-----+-------+----+----------+
|   tanggal|tahun|bulan|kuartal|hari|nama_bulan|
+----------+-----+-----+-------+----+----------+
|2026-05-02| 2026|    5|      2|   2|       May|
|2026-05-01| 2026|    5|      2|   1|       May|
|2026-05-04| 2026|    5|      2|   4|       May|
|2026-05-03| 2026|    5|      2|   3|       May|
+----------+-----+-----+-------+----+----------+



In [7]:
fact_sales = silver_df

In [8]:
fact_sales = fact_sales.join(
    dim_produk,
    ["produk", "kategori"]
)

In [9]:
fact_sales = fact_sales.join(
    dim_kota,
    ["kota"]
)

In [10]:
from pyspark.sql.functions import col

fact_sales = fact_sales.withColumn(
    "revenue",
    col("jumlah") * col("harga")
)

In [11]:
fact_sales = fact_sales.select(
    "id",
    "tanggal",
    "produk_key",
    "kota_key",
    "jumlah",
    "harga",
    "revenue"
)

fact_sales.show()

+---+----------+----------+--------+------+-------+-------+
| id|   tanggal|produk_key|kota_key|jumlah|  harga|revenue|
+---+----------+----------+--------+------+-------+-------+
|  1|2026-05-01|         4|       2|     1|8000000|8000000|
|  2|2026-05-01|         1|       4|     2| 150000| 300000|
|  3|2026-05-02|         3|       3|     3|  75000| 225000|
|  4|2026-05-02|         4|       2|     1|8000000|8000000|
|  5|2026-05-03|         2|       1|    10|   5000|  50000|
|  6|2026-05-03|         4|       2|     1|8000000|8000000|
|  7|2026-05-04|         1|       4|     1| 150000| 150000|
|  8|2026-05-04|         3|       3|     2|  75000| 150000|
+---+----------+----------+--------+------+-------+-------+



In [12]:
dim_produk.write \
    .mode("overwrite") \
    .saveAsTable(
        "warehouse.dim_produk_star"
    )

In [13]:
dim_kota.write \
    .mode("overwrite") \
    .saveAsTable(
        "warehouse.dim_kota"
    )

In [15]:
dim_tanggal.write \
    .mode("overwrite") \
    .saveAsTable(
        "warehouse.dim_tanggal"
    )

In [16]:
fact_sales.write \
    .mode("overwrite") \
    .saveAsTable(
        "warehouse.fact_sales_star"
    )

In [17]:
spark.sql("""
SHOW TABLES IN warehouse
""").show(truncate=False)

+---------+-------------------+-----------+
|namespace|tableName          |isTemporary|
+---------+-------------------+-----------+
|warehouse|dim_kota           |false      |
|warehouse|dim_produk         |false      |
|warehouse|dim_produk_star    |false      |
|warehouse|dim_tanggal        |false      |
|warehouse|fact_sales         |false      |
|warehouse|fact_sales_star    |false      |
|warehouse|gold_revenue_harian|false      |
|warehouse|gold_revenue_kota  |false      |
|warehouse|gold_revenue_produk|false      |
|warehouse|revenue_kategori   |false      |
|warehouse|revenue_produk     |false      |
+---------+-------------------+-----------+

